In [1]:
### model training
# results in google drive ('AD classification training')

## TODO ##
# finalize classification system
# [ ] decide which channels to experiment with (which are important for AD classification?) + put in all biomarkers => should be better or same
# [ ] make sure it works with any type of input (ch size) and returns AD prognosis + confidence (test set)

# improve classification
# [ ] more data
# [ ] change feature extraction and see if it works better with consistent in channels (setting the rest to 0)
# [ ] make 2nd model that returns output from log reg
# [ ] incorporate counterfactual

# test
# [ ] confusion matrices for tests



# 1. training data preparation. creates an excel sheet:
# ID | diagnosis (GT) | num_label (GT) | gender | age | mmse | path_to_eeg | path_to_feature_vector | path_to_small_feature_vector | biomarkers

# 2. train simple classification models on feature vector + biographics + biomarkers > ad diagnosis

# 3. compare embedding model, small embedding model, tabular model, hybrid model, and small hybrid model
#    alos check that the problem is not too trivial to involve training

import torch, gc

gc.collect()                   # clear Python garbage
torch.cuda.empty_cache()       # clear PyTorch cache

In [1]:
#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("rifahramisa/eye-open-eeg-alzheimers")

#print("Path to dataset files:", path)

100%|██████████| 706M/706M [00:17<00:00, 42.7MB/s] 

Extracting files...


Path to dataset files: /home/jans26/.cache/kagglehub/datasets/rifahramisa/eye-open-eeg-alzheimers/versions/1


In [3]:
### 1. prepare training data

import csv
import pandas as pd
from utility import save_eeg_features_from_path, save_selected_eeg_features_from_path, get_biomarkers_from_path

# path to participants.tsv (from kaggle open nuro dataset)
path1 = '/home/jans26/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/participants.tsv'
#path1 = 'C:/Users/marks/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/participants.tsv'
path2 = '/home/jans26/.cache/kagglehub/datasets/rifahramisa/eye-open-eeg-alzheimers/versions/1/eye-open-dataset/participants.tsv'
#path2 = 'C:/Users/marks/.cache/kagglehub/datasets/rifahramisa/eye-open-eeg-alzheimers/versions/1/eye-open-dataset/participants.tsv'

label2num = {
    'C': 0,    # healthy control
    'A': 1,    # alzheimer’s disease 
    'F': 2     # frontotemporal dementia
}

# go thru all patients in tsv and save their data
data = []
with open(path1, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        eeg_id = row['participant_id']
        print(eeg_id)
        diagnosis = row['Group']
        num_label = label2num[row['Group']]
        gender = row['Gender']
        gender_bin = 1 if gender == 'M' else 0
        age = row['Age']
        mmse = row['MMSE']
        path_to_eeg_closed = f'/home/jans26/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/derivatives/{eeg_id}/eeg/{eeg_id}_task-eyesclosed_eeg.set'
        #path_to_eeg_closed = f'C:/Users/marks/.cache/kagglehub/datasets/yosftag/open-nuro-dataset/versions/1/dataset/derivatives/{eeg_id}/eeg/{eeg_id}_task-eyesclosed_eeg.set'
        path_to_eeg_open = f'/home/jans26/.cache/kagglehub/datasets/rifahramisa/eye-open-eeg-alzheimers/versions/1/eye-open-dataset/{eeg_id}/eeg/{eeg_id}_task-photomark_eeg.set'
        #path_to_eeg_open =  f'C:/Users/marks/.cache/kagglehub/datasets/rifahramisa/eye-open-eeg-alzheimers/versions/1/eye-open-dataset/{eeg_id}/eeg/{eeg_id}_task-photomark_eeg.set'
        
        # compute and save feature vector, save path in data
        path_to_feature_vector_closed = save_eeg_features_from_path(path_to_eeg_closed, 'eegpt_features_closed', eeg_id)
        path_to_small_feature_vector_closed = save_selected_eeg_features_from_path(path_to_eeg_closed, 'eegpt_features_closed', eeg_id)
        
        path_to_feature_vector_open = save_eeg_features_from_path(path_to_eeg_open, 'eegpt_features_open', eeg_id)
        path_to_small_feature_vector_open = save_selected_eeg_features_from_path(path_to_eeg_open, 'eegpt_features_open', eeg_id)
        
        # compute and save biomarkers
        alpha, beta, theta, delta, theta_alpha_ratio, slowing_index, coherence, entropy_val = get_biomarkers_from_path(path_to_eeg_closed)
        alpha2, beta2, theta2, delta2, theta_alpha_ratio2, slowing_index2, coherence2, entropy_val2 = get_biomarkers_from_path(path_to_eeg_open)
        
        data.append([
            eeg_id, diagnosis, num_label, gender, gender_bin, age, mmse, 
            path_to_eeg_closed, path_to_eeg_open, 
            path_to_feature_vector_closed, path_to_feature_vector_open, 
            path_to_small_feature_vector_closed, path_to_small_feature_vector_open,
            alpha, beta, theta, delta, theta_alpha_ratio, slowing_index, coherence, entropy_val,
            alpha2, beta2, theta2, delta2, theta_alpha_ratio2, slowing_index2, coherence2, entropy_val2
        ])

columns = [
    "eeg_id", "diagnosis", "num_label", "gender", "gender_bin", "age", "mmse",
    "path_to_eeg_closed", "path_to_eeg_open",
    "path_to_feature_vector_closed", "path_to_feature_vector_open",
    "path_to_small_feature_vector_closed", "path_to_small_feature_vector_open", 
    "alpha", "beta", "theta", "delta", "theta_alpha_ratio", "slowing_index", "coherence", "entropy",
    "alpha2", "beta2", "theta2", "delta2", "theta_alpha_ratio2", "slowing_index2", "coherence2", "entropy2"
]

df = pd.DataFrame(data, columns=columns)

df.to_excel("eeg_dataset_with_eye-open.xlsx", index=False)

sub-001
sub-002
sub-003
sub-004
sub-005
sub-006
sub-007
sub-008
sub-009
sub-010
sub-011
sub-012
sub-013
sub-014
sub-015
sub-016
sub-017
sub-018
sub-019
sub-020
sub-021
sub-022
sub-023
sub-024
sub-025
sub-026
sub-027
sub-028
sub-029
sub-030
sub-031
sub-032
sub-033
sub-034
sub-035
sub-036
sub-037
sub-038
sub-039
sub-040
sub-041
sub-042
sub-043
sub-044
sub-045
sub-046
sub-047
sub-048
sub-049
sub-050
sub-051
sub-052
sub-053
sub-054
sub-055
sub-056
sub-057
sub-058
sub-059
sub-060
sub-061
sub-062
sub-063
sub-064
sub-065
sub-066
sub-067
sub-068
sub-069
sub-074
sub-075
sub-076
sub-077
sub-078
sub-079
sub-080
sub-081
sub-082
sub-083
sub-084
sub-085
sub-086
sub-087
sub-088


In [5]:
### 2. training classifier
# dataset that can give me the features and label
# simple nn for classification

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
import torch.optim as optim
import pandas as pd
import os

# models
class EmbeddingClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, embed_vec):
        return self.net(embed_vec)

class TabularClassifier(nn.Module):
    def __init__(self, num_tabular, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(num_tabular, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, tabular_vec):
        return self.net(tabular_vec)

class HybridClassifier(nn.Module):
    def __init__(self, num_tabular, num_classes):
        super().__init__()

        # -------- Embedding branch (512 → 128)
        self.embed_net = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        # -------- Tabular branch (~10 → 64)
        self.tabular_net = nn.Sequential(
            nn.Linear(num_tabular, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        # -------- Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 + 64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

        # Learnable scaling (helps prevent tabular being ignored)
        self.alpha = nn.Parameter(torch.tensor(1.0))

    def forward(self, embed_vec, tabular_vec):
        x1 = self.embed_net(embed_vec)       # [B, 128]
        x2 = self.tabular_net(tabular_vec)   # [B, 64]

        x = torch.cat([x1, self.alpha * x2], dim=1)
        return self.classifier(x)

# datasets
class EmbeddingDataset(Dataset):
    def __init__(
        self,
        excel_path,
        label_col,
        embed_col
    ):
        self.df = pd.read_excel(excel_path)

        self.labels = self.df[label_col].values.astype("int64")
        self.embed_paths = self.df[embed_col].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        embed_path = self.embed_paths[idx]
        embed = torch.load(embed_path)

        if embed.ndim > 1:
            embed = embed.squeeze()

        embed = embed.float()
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return embed, label

class TabularDataset(Dataset):
    def __init__(
        self,
        excel_path,
        feature_cols,
        label_col,
        normalize=True
    ):
        self.df = pd.read_excel(excel_path)

        self.features = self.df[feature_cols].values.astype("float32")
        self.labels = self.df[label_col].values.astype("int64")

        self.normalize = normalize
        if normalize:
            self.mean = self.features.mean(axis=0)
            self.std = self.features.std(axis=0) + 1e-8

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        tab = self.features[idx]

        if self.normalize:
            tab = (tab - self.mean) / self.std

        tab = torch.tensor(tab, dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return tab, label

class HybridDataset(Dataset):
    def __init__(
        self,
        excel_path,
        feature_cols,
        label_col,
        embed_col,
        normalize=True
    ):
        self.df = pd.read_excel(excel_path)

        self.feature_cols = feature_cols
        self.label_col = label_col
        self.embed_col = embed_col

        # ---- Convert to numpy for speed
        self.features = self.df[feature_cols].values.astype("float32")
        self.labels = self.df[label_col].values.astype("int64")
        self.embed_paths = self.df[embed_col].values

        # ---- Normalization (VERY important)
        self.normalize = normalize
        if normalize:
            self.mean = self.features.mean(axis=0)
            self.std = self.features.std(axis=0) + 1e-8

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # ---- Load embedding
        embed_path = self.embed_paths[idx]
        embed = torch.load(embed_path)  # assumes tensor [512]

        # Safety: ensure correct shape
        if embed.ndim > 1:
            embed = embed.squeeze()

        embed = embed.float()

        # ---- Tabular features
        tab = self.features[idx]

        if self.normalize:
            tab = (tab - self.mean) / self.std

        tab = torch.tensor(tab, dtype=torch.float32)

        # ---- Label
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return embed, tab, label

        
# training + evaluation
def train_embedding(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for embed, y in loader:
        embed = embed.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(embed)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def train_tabular(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for tab, y in loader:
        tab = tab.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(tab)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)
    
def train_hybrid(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for embed, tab, y in loader:
        embed = embed.to(device)
        tab = tab.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(embed, tab)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)
    
@torch.no_grad()
def evaluate_hybrid(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        x1, x2, y = batch
        x1 = x1.to(device)
        x2 = x2.to(device)
        y = y.to(device)

        logits = model(x1, x2)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

    return correct / total

@torch.no_grad()
def evaluate_simple(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        x, y = batch
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

    return correct / total

# ============================================
# setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_tabular = 11     # your float features (including the binary one)
num_classes = 3      # or 2

embedding_model = EmbeddingClassifier(num_classes).to(device)
small_embedding_model = EmbeddingClassifier(num_classes).to(device)
tabular_model = TabularClassifier(num_tabular, num_classes).to(device)
hybrid_model = HybridClassifier(num_tabular, num_classes).to(device)
small_hybrid_model = HybridClassifier(num_tabular, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
embedding_optimizer = optim.Adam(embedding_model.parameters(), lr=1e-3)
small_embedding_optimizer = optim.Adam(small_embedding_model.parameters(), lr=1e-3)
tabular_optimizer = optim.Adam(tabular_model.parameters(), lr=1e-3)
hybrid_optimizer = optim.Adam(hybrid_model.parameters(), lr=1e-3)
small_hybrid_optimizer = optim.Adam(small_hybrid_model.parameters(), lr=1e-3)

feature_cols = [
    "gender_bin",
    "age", 
    "mmse",
    "alpha2", 
    "beta2", 
    "theta2", 
    "delta2",
    "theta_alpha_ratio2", 
    "slowing_index2", 
    "coherence2", 
    "entropy2"
]

embedding_dataset = EmbeddingDataset(
    excel_path="eeg_dataset_with_eye-open.xlsx",
    label_col="num_label",
    embed_col="path_to_feature_vector_open"
)

small_embedding_dataset = EmbeddingDataset(
    excel_path="eeg_dataset_with_eye-open.xlsx",
    label_col="num_label",
    embed_col="path_to_small_feature_vector_open"
)

tabular_dataset = TabularDataset(
    excel_path="eeg_dataset_with_eye-open.xlsx",
    feature_cols=feature_cols,
    label_col="num_label",
)

hybrid_dataset = HybridDataset(
    excel_path="eeg_dataset_with_eye-open.xlsx",
    feature_cols=feature_cols,
    label_col="num_label",
    embed_col="path_to_feature_vector_open"
)

small_hybrid_dataset = HybridDataset(
    excel_path="eeg_dataset_with_eye-open.xlsx",
    feature_cols=feature_cols,
    label_col="num_label",
    embed_col="path_to_small_feature_vector_open"
)

assert len(embedding_dataset) == len(small_embedding_dataset) == len(tabular_dataset) == len(hybrid_dataset) == len(small_hybrid_dataset)
#print(f"Dataset sizes: {len(embedding_dataset)} | {len(small_embedding_dataset)} | {len(tabular_dataset)} | {len(hybrid_dataset)} | {len(small_hybrid_dataset)}")

train_size = int(0.75 * len(embedding_dataset))
val_size = len(embedding_dataset) - train_size
batch_size = 11

embedding_train_dataset, embedding_val_dataset = random_split(embedding_dataset, [train_size, val_size])
small_embedding_train_dataset, small_embedding_val_dataset = random_split(small_embedding_dataset, [train_size, val_size])
tabular_train_dataset, tabular_val_dataset = random_split(tabular_dataset, [train_size, val_size])
hybrid_train_dataset, hybrid_val_dataset = random_split(hybrid_dataset, [train_size, val_size])
small_hybrid_train_dataset, small_hybrid_val_dataset = random_split(small_hybrid_dataset, [train_size, val_size])

embedding_train_loader = DataLoader(embedding_train_dataset, batch_size=batch_size, shuffle=True)
embedding_val_loader = DataLoader(embedding_val_dataset, batch_size=batch_size)

small_embedding_train_loader = DataLoader(small_embedding_train_dataset, batch_size=batch_size, shuffle=True)
small_embedding_val_loader = DataLoader(small_embedding_val_dataset, batch_size=batch_size)

tabular_train_loader = DataLoader(tabular_train_dataset, batch_size=batch_size, shuffle=True)
tabular_val_loader = DataLoader(tabular_val_dataset, batch_size=batch_size)

hybrid_train_loader = DataLoader(hybrid_train_dataset, batch_size=batch_size, shuffle=True)
hybrid_val_loader = DataLoader(hybrid_val_dataset, batch_size=batch_size)

small_hybrid_train_loader = DataLoader(small_hybrid_train_dataset, batch_size=batch_size, shuffle=True)
small_hybrid_val_loader = DataLoader(small_hybrid_val_dataset, batch_size=batch_size)

# training loop

epochs = 50
best_acc = {
    'embedding': 0,
    'small_embedding': 0,
    'tabular': 0,
    'hybrid': 0,
    'small_hybrid': 0
}
for epoch in range(epochs):
    embedding_train_loss = train_embedding(embedding_model, embedding_train_loader, embedding_optimizer, criterion, device)
    small_embedding_train_loss = train_embedding(small_embedding_model, small_embedding_train_loader, small_embedding_optimizer, criterion, device)
    tabular_train_loss = train_tabular(tabular_model, tabular_train_loader, tabular_optimizer, criterion, device)
    hybrid_train_loss = train_hybrid(hybrid_model, hybrid_train_loader, hybrid_optimizer, criterion, device)
    small_hybrid_train_loss = train_hybrid(small_hybrid_model, small_hybrid_train_loader, small_hybrid_optimizer, criterion, device)
    
    embedding_val_acc = evaluate_simple(embedding_model, embedding_val_loader, device)
    small_embedding_val_acc = evaluate_simple(small_embedding_model, small_embedding_val_loader, device)
    tabular_val_acc = evaluate_simple(tabular_model, tabular_val_loader, device)
    hybrid_val_acc = evaluate_hybrid(hybrid_model, hybrid_val_loader, device)
    small_hybrid_val_acc = evaluate_hybrid(small_hybrid_model, small_hybrid_val_loader, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: (E){embedding_train_loss:.4f} | (Es){small_embedding_train_loss:.4f} | (T){tabular_train_loss:.4f} | (H){hybrid_train_loss:.4f} | (Hs){small_hybrid_train_loss:.4f}")
    print(f"Val Acc: (E){embedding_val_acc:.4f} | (Es){small_embedding_val_acc:.4f} | (T){tabular_val_acc:.4f} | (H){hybrid_val_acc:.4f} | (Hs){small_hybrid_val_acc:.4f}")
    print("-" * 40)

    if(embedding_val_acc > best_acc['embedding']):
        best_acc['embedding'] = embedding_val_acc
        torch.save(embedding_model.state_dict(), "eeg_classification_model_embedding.pt")
    if(small_embedding_val_acc > best_acc['small_embedding']):
        best_acc['small_embedding'] = small_embedding_val_acc
        torch.save(small_embedding_model.state_dict(), "eeg_classification_model_small_embedding.pt")
    if(tabular_val_acc > best_acc['tabular']):
        best_acc['tabular'] = tabular_val_acc
        torch.save(tabular_model.state_dict(), "eeg_classification_model_tabular.pt")
    if(hybrid_val_acc > best_acc['hybrid']):
        best_acc['hybrid'] = hybrid_val_acc
        torch.save(hybrid_model.state_dict(), "eeg_classification_model_hybrid.pt")
    if(small_hybrid_val_acc > best_acc['small_hybrid']):
        best_acc['small_hybrid'] = small_hybrid_val_acc
        torch.save(small_hybrid_model.state_dict(), "eeg_classification_model_small_hybrid.pt")

print(f"Best Model Accuracies:\nEmbedding: {best_acc['embedding']:.4f}\nSmall Embedding: {best_acc['small_embedding']:.4f}\nTabular: {best_acc['tabular']:.4f}\nHybrid: {best_acc['hybrid']:.4f}\nSmall Hybrid: {best_acc['small_hybrid']:.4f}")


Epoch 1/50
Train Loss: (E)1.1059 | (Es)1.1051 | (T)1.1317 | (H)1.0961 | (Hs)1.0915
Val Acc: (E)0.3636 | (Es)0.2727 | (T)0.4091 | (H)0.4091 | (Hs)0.5000
----------------------------------------
Epoch 2/50
Train Loss: (E)1.0872 | (Es)1.0686 | (T)1.0823 | (H)1.0815 | (Hs)1.0384
Val Acc: (E)0.3636 | (Es)0.2727 | (T)0.7727 | (H)0.4091 | (Hs)0.5455
----------------------------------------
Epoch 3/50
Train Loss: (E)1.0826 | (Es)1.0621 | (T)1.0320 | (H)1.0600 | (Hs)1.0107
Val Acc: (E)0.3636 | (Es)0.2727 | (T)0.5909 | (H)0.4091 | (Hs)0.5455
----------------------------------------
Epoch 4/50
Train Loss: (E)1.0858 | (Es)1.0642 | (T)0.9958 | (H)1.0442 | (Hs)0.9543
Val Acc: (E)0.3636 | (Es)0.2727 | (T)0.6364 | (H)0.4545 | (Hs)0.5455
----------------------------------------
Epoch 5/50
Train Loss: (E)1.0827 | (Es)1.0612 | (T)0.9506 | (H)1.0097 | (Hs)0.8870
Val Acc: (E)0.3636 | (Es)0.2727 | (T)0.6364 | (H)0.5455 | (Hs)0.5455
----------------------------------------
Epoch 6/50
Train Loss: (E)1.0790 | 

In [6]:
# 3. baseline compasrison

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

def loader2numpy(loader, device="cpu"):
    X_list = []
    y_list = []

    for load in loader:
        if len(load) == 3:
            x1, x2, y = load
            x1 = x1.detach().cpu()
            x2 = x2.detach().cpu()
            x = torch.cat([x1, x2], dim=1)
        else:
            x, y = load
            x = x.detach().cpu()
        y = y.detach().cpu()

        X_list.append(x)
        y_list.append(y)

    # concatenate all batches
    X = torch.cat(X_list, dim=0).numpy()
    y = torch.cat(y_list, dim=0).numpy()

    return X, y
    
# go thru all 5 options and calculate all for them
train_val_loaders = [
    [embedding_train_loader, embedding_val_loader],
    [small_embedding_train_loader, small_embedding_val_loader],
    [tabular_train_loader, tabular_val_loader],
    [hybrid_train_loader, hybrid_val_loader],
    [small_hybrid_train_loader, small_hybrid_val_loader],
]

results = [['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'MLP Classifier']]

for train_loader, val_loader in train_val_loaders:
    X_train, y_train = loader2numpy(train_loader)
    X_val, y_val = loader2numpy(val_loader)

    # A Logistic Regression
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    lr_acc = accuracy_score(y_val, lr.predict(X_val))

    # B Random Forest
    rf = RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)
    rf_acc = accuracy_score(y_val, rf.predict(X_val))

    # C Gradient Boosting (STRONG baseline)
    gb = HistGradientBoostingClassifier(max_iter=200)
    gb.fit(X_train, y_train)
    gb_acc = accuracy_score(y_val, gb.predict(X_val))

    # D MLP (sklearn NN baseline)
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=200
    )
    mlp.fit(X_train, y_train)
    mlp_acc = accuracy_score(y_val, mlp.predict(X_val))

    results.append([lr_acc, rf_acc, gb_acc, mlp_acc])


print("\n=== BASELINE COMPARISON  (E, Es, T, H, Hs) ===")
for lr, rf, gb, mlp in results:
    print(f"{lr} | {rf} | {gb} | {mlp}")

/home/jans26/.conda/envs/myenv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jans26/.conda/envs/myenv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/jans26/.conda/envs/myenv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



=== BASELINE COMPARISON  (E, Es, T, H, Hs) ===
Logistic Regression | Random Forest | Gradient Boosting | MLP Classifier
0.36363636363636365 | 0.2727272727272727 | 0.2727272727272727 | 0.36363636363636365
0.22727272727272727 | 0.2727272727272727 | 0.22727272727272727 | 0.22727272727272727
0.8181818181818182 | 0.9090909090909091 | 0.8181818181818182 | 0.8636363636363636
0.7727272727272727 | 0.45454545454545453 | 0.8181818181818182 | 0.8181818181818182
0.8181818181818182 | 0.5454545454545454 | 0.7272727272727273 | 0.6363636363636364


/home/jans26/.conda/envs/myenv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
